<a href="https://colab.research.google.com/github/kasturikirankumar1101-lab/AI_TOOLS/blob/main/RAGretrivalandresponsewithChatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# One shot for full setup. Please run this commands(each step once) before you run the above programs

# Step1. After this install, restart the session.
!pip install -U pip setuptools wheel

# Step2. After this install, restart the session.
!pip install -q \
  langchain==0.2.16 \
  langchain-community==0.2.16 \
  langchain-openai==0.1.23 \
  langchain-chroma==0.1.4 \
  chromadb==0.4.24 \
  langchain-text-splitters==0.2.2 \
  pypdf \
  pydantic==2.12.3 \
  aiofiles==24.1.0 \
  requests==2.32.4

# Step 3. Install and restart the session
!pip install numpy==1.26.4 --force-reinstall

# Step4. install and restrt the session

!apt-get install -y poppler-utils

# Step5. Run the below code to restart the session and impact all the above installs
import os
os.kill(os.getpid(), 9)

# Step6. To mount the drive to access the input files.
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
# chatbot.py
# This program uses the chatbot instead of chat
# This part of the program would create the vector retrieval and share the data to
# LLMs to respond to the user queries.
import logging
from google.colab import userdata as colab_userdata
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from chromadb.config import Settings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

# ── Logging setup ──────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# ── API Key from Google Colab Secrets ─────────────────────────────────────────
try:
    api_key = colab_userdata.get("OPENAI_API_KEY").strip()
    logger.info("✅ API key loaded from Colab Secrets.")
except Exception as e:
    raise EnvironmentError(
        f"Failed to load OPENAI_API_KEY from Colab Secrets: {e}\n"
        "👉 Open the 🔑 Secrets panel (left sidebar) and add OPENAI_API_KEY."
    ) from e

if not api_key.startswith("sk-"):
    logger.warning("OPENAI_API_KEY does not start with 'sk-' — double-check the value.")

# ── Configuration ──────────────────────────────────────────────────────────────
CHROMA_DIR = "/content/chroma_db"
COLLECTION_NAME = "rag_chatbot"
TOP_K = 5

# ── Vector Store ───────────────────────────────────────────────────────────────
def get_vector_store():
    """Connect to existing ChromaDB vector store."""
    embeddings = OpenAIEmbeddings(
        api_key=api_key,
        model="text-embedding-3-small",
        dimensions=1536
    )
    return Chroma(
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DIR,
        embedding_function=embeddings

    )

# ── Document Formatter ─────────────────────────────────────────────────────────
def format_docs(docs):
    """Format retrieved documents into a single context string."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        print("Is it coming here")
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "N/A")
        formatted.append(
            f"[Source {i}: {source}, Page {page}]\n{doc.page_content}"
        )
    print("ATTENTION PLEAE!!!!!!!!")
    print(f"data = {docs}")
    return "\n\n---\n\n".join(formatted)

# ── RAG Prompt ─────────────────────────────────────────────────────────────────
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions
based on the provided context. Follow these rules strictly:

1. Only answer based on the provided context
2. If the context does not contain enough information, say so
3. Cite your sources using [Source N] notation
4. Be concise but thorough
5. If asked about something outside the context, explain that
   your knowledge is limited to the provided documents

Context:
{context}"""),
    ("human", "{question}")
])

# ── RAG Chain Builder ──────────────────────────────────────────────────────────
def build_rag_chain():
    """Build the complete RAG chain."""
    vector_store = get_vector_store()
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": TOP_K, "fetch_k": 20}
    )

    llm = ChatOpenAI(
        api_key=api_key,
        model="gpt-4o",
        temperature=0.1,
        max_tokens=2048
    )

    chain = (
        {"context": retriever | format_docs,
         "question": RunnablePassthrough()}
        | RAG_PROMPT
        | llm
        | StrOutputParser()
    )
    return chain

# ── Startup ────────────────────────────────────────────────────────────────────
logger.info("Initializing RAG chain...")
rag_chain = build_rag_chain()
logger.info("RAG Chatbot Ready!")

# ── Gradio Respond Function ────────────────────────────────────────────────────
def respond(message: str, history: list) -> str:
    """
    Gradio ChatInterface-compatible respond function.

    Args:
        message: The current user message.
        history: List of prior [user, assistant] message pairs (managed by Gradio).

    Returns:
        The assistant's response string.
    """
    if not message.strip():
        return "Please enter a question."

    try:
        response = rag_chain.invoke(message)
        return response
    except Exception as e:
        logger.error("Error invoking RAG chain: %s", e)
        return f"⚠️ An error occurred while processing your request: {e}"

# ── Gradio UI ──────────────────────────────────────────────────────────────────
demo = gr.ChatInterface(
    fn=respond,
    title="📚 RAG Chatbot",
    description=(
        "Ask questions about your documents. "
        "The assistant will answer strictly based on the ingested content and cite its sources."
    ),
    examples=[
        "What is this document about?",
        "Can you summarize the key points?",
        "What are the main topics covered?",
    ],
    theme=gr.themes.Soft(),
    chatbot=gr.Chatbot(height=500, render_markdown=True),
)

# ── Entry Point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    demo.launch(share=False,debug=True)